In [1]:
import itertools
import pickle
from collections.abc import Iterator
from pathlib import Path
from typing import Protocol

import supersuit as ss
from pettingzoo.atari import tennis_v3

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import torch
from torch import nn, optim


# Agent Definitions

## Base Agent Class

In [2]:
class Agent:
    """Base class for reinforcement learning agents."""

    def __init__(self, seed: int, action_space: int) -> None:
        """Initialize the agent with its action space."""
        self.action_space = action_space
        self.rng = np.random.default_rng(seed = seed)

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> None:
        """Select an action from the current observation."""

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update agent parameters from one transition."""

    def save(self, filename: str) -> None:
        """Save the agent state to disk."""
        with Path(filename).open("wb") as f:
            pickle.dump(self.__dict__, f)

    def load(self, filename: str) -> None:
        """Load the agent state from disk."""
        with Path(filename).open("rb") as f:
            self.__dict__.update(pickle.load(f))  # noqa: S301


## Random Agent

In [3]:
class RandomAgent(Agent):
    """A simple agent that selects actions randomly."""

    def get_action(  # type: ignore[override]  # pyright: ignore[reportIncompatibleMethodOverride]
        self,
        observation: np.ndarray,
        epsilon: float = 0.0,
    ) -> int:
        """Select an action randomly, ignoring the observation and epsilon."""
        _ = observation
        _ = epsilon
        return int(self.rng.integers(0, self.action_space))


## Torch Agent

In [4]:
class TorchAgent(Agent):
    """A reinforcement-learning agent using a PyTorch model to estimate action values."""

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        lr: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize TorchAgent.

        Args:
            n_features: Number of input features (state dimensionality).
            n_actions: Number of possible discrete actions.
            lr: Learning rate (kept for subclasses).
            gamma: Discount factor.
            seed: RNG seed for reproducibility.

        """
        # Initialize base Agent with seed and action space size
        super().__init__(seed, n_actions)

        # keep parameters to avoid unused-argument warnings and for subclasses
        self.n_features = n_features
        self.lr = lr

        self.gamma = gamma
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model: nn.Module | None = None
        self.optimizer: optim.Optimizer | None = None
        self.criterion = nn.MSELoss()

    def save(self, filename: str) -> None:
        """Save model parameters to disk."""
        if self.model is None:
            message = "Model is not initialized. Define self.model before calling save()."
            raise ValueError(message)
        torch.save(self.model.state_dict(), filename)

    def load(self, filename: str) -> None:
        """Load model parameters from disk into existing model."""
        if self.model is None:
            message = "Model is not initialized. Define self.model before calling load()."
            raise ValueError(message)
        state = torch.load(filename, map_location=self.device)
        self.model.load_state_dict(state)
        self.model.eval()

    def get_action(  # type: ignore[override]  # pyright: ignore[reportIncompatibleMethodOverride]
        self,
        observation: np.ndarray,
        epsilon: float = 0.0,
    ) -> int:
        """Return an action for the given observation using epsilon-greedy policy."""
        if self.model is None:
            message = "Model is not initialized. Define self.model before calling get_action()."
            raise ValueError(message)

        if self.rng.random() < epsilon:
            return int(self.rng.integers(0, self.action_space))

        state_t = (
            torch.as_tensor(observation, dtype=torch.float32, device=self.device)
            / 255.0
        )
        if state_t.ndim == 1:
            state_t = state_t.unsqueeze(0)

        self.model.eval()
        with torch.no_grad():
            q_values = self.model(state_t).squeeze(0)

        return int(torch.argmax(q_values).item())


## Linear Agent

In [5]:
class LinearAgent(TorchAgent):
    """An agent that uses a simple linear model to estimate action values."""

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        lr: float = 0.001,
        gamma: float = 0.99,
        method: str = "q_learning",
    ) -> None:
        """Initialize LinearAgent with a linear model and specified update method."""
        super().__init__(n_features, n_actions, lr, gamma)
        self.method = method
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features, n_actions),
        ).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update the agent's value estimates using either Q-learning or SARSA."""
        if self.model is None:
            message = "Model is not initialized."
            raise ValueError(message)
        if self.optimizer is None:
            message = "Optimizer is not initialized."
            raise ValueError(message)

        model = self.model
        optimizer = self.optimizer

        state_t = torch.as_tensor(state, device=self.device, dtype=torch.float32).unsqueeze(0) / 255.0
        next_state_t = torch.as_tensor(next_state, device=self.device, dtype=torch.float32).unsqueeze(0) / 255.0
        reward_t = torch.as_tensor(reward, device=self.device, dtype=torch.float32)
        done_t = torch.as_tensor(float(done), device=self.device, dtype=torch.float32)

        q_curr = model(state_t).squeeze(0)[action]

        with torch.no_grad():
            if done:
                q_next = torch.tensor(0.0, device=self.device)
            elif self.method == "q_learning":
                q_next = model(next_state_t).squeeze(0).max()
            elif self.method == "sarsa":
                if next_action is None:
                    message = "next_action is required for SARSA updates."
                    raise ValueError(message)
                q_next = model(next_state_t).squeeze(0)[next_action]
            else:
                message = (
                    f"Unknown method: {self.method}. Supported methods are "
                    "'q_learning' and 'sarsa'."
                )
                raise ValueError(message)

            target = reward_t + (1.0 - done_t) * self.gamma * q_next

        loss = self.criterion(q_curr, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


## DQN Agent

In [6]:
class DQNAgent(TorchAgent):
    """An agent that uses a deep neural network to estimate action values, with support for experience replay and target networks."""

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        lr: float = 0.0001,
        gamma: float = 0.99,
        buffer_size: int = 10000,
        batch_size: int = 64,
        target_update_freq: int = 1000,
    ) -> None:
        """Initialize DQNAgent with a deep neural network, experience replay buffer, and target network."""
        super().__init__(n_features, n_actions, lr, gamma)
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.update_step = 0

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
        ).to(self.device)

        self.target_model: nn.Module = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
        ).to(self.device)
        self.target_model.load_state_dict(self.model.state_dict())

        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.replay_buffer: list[tuple[np.ndarray, int, float, np.ndarray, bool]] = []

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update the agent's value estimates using experience replay and target network."""
        _ = next_action  # kept for API compatibility with other agents

        if self.model is None:
            message = "Model is not initialized."
            raise ValueError(message)
        if self.optimizer is None:
            message = "Optimizer is not initialized."
            raise ValueError(message)

        model = self.model
        optimizer = self.optimizer
        target_model = self.target_model

        self.replay_buffer.append((state, action, reward, next_state, done))
        if len(self.replay_buffer) > self.buffer_size:
            self.replay_buffer.pop(0)

        if len(self.replay_buffer) < self.batch_size:
            return

        idx = self.rng.choice(
            len(self.replay_buffer), size=self.batch_size, replace=False,
        )
        batch = [self.replay_buffer[i] for i in idx]

        states_np, actions_np, rewards_np, next_states_np, dones_np = map(
            np.array, zip(*batch, strict=True),
        )

        states_t = torch.as_tensor(states_np, dtype=torch.float32, device=self.device) / 255.0
        next_states_t = torch.as_tensor(next_states_np, dtype=torch.float32, device=self.device) / 255.0
        actions_t = torch.as_tensor(actions_np, dtype=torch.long, device=self.device)
        rewards_t = torch.as_tensor(rewards_np, dtype=torch.float32, device=self.device)
        dones_t = torch.as_tensor(
            dones_np.astype(float), dtype=torch.float32, device=self.device,
        )

        q_curr = model(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            q_next = target_model(next_states_t).max(dim=1)[0]
            target = rewards_t + (1.0 - dones_t) * self.gamma * q_next

        loss = self.criterion(q_curr, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        self.update_step += 1
        if self.update_step % self.target_update_freq == 0:
            target_model.load_state_dict(model.state_dict())


## Monte Carlo Agent

In [7]:
class MonteCarloAgent(LinearAgent):
    """An agent that uses Monte Carlo methods to estimate action values."""

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        lr: float = 0.001,
        gamma: float = 0.99,
    ) -> None:
        """Initialize MonteCarloAgent."""
        super().__init__(n_features, n_actions, lr, gamma, method="monte_carlo")
        self.episode_buffer: list[tuple[np.ndarray, int, float]] = []

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update value estimates with Monte Carlo returns at episode end."""
        _ = next_state
        _ = next_action

        if self.model is None:
            msg = "Model is not initialized."
            raise ValueError(msg)
        if self.optimizer is None:
            msg = "Optimizer is not initialized."
            raise ValueError(msg)

        model = self.model
        optimizer = self.optimizer

        self.episode_buffer.append((state, action, reward))

        if not done:
            return

        returns = 0.0
        for s, a, r in reversed(self.episode_buffer):
            returns = self.gamma * returns + r

            state_t = torch.as_tensor(s, dtype=torch.float32, device=self.device)
            if state_t.ndim == 1:
                state_t = state_t.unsqueeze(0)

            q_val = model(state_t).squeeze(0)[a]
            target = torch.as_tensor(returns, dtype=torch.float32, device=self.device)

            loss = self.criterion(q_val, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        self.episode_buffer = []


# Tennis Environment

In [8]:
def create_env(obs_type: str = "ram"):
    """Create the Tennis environment with optional preprocessing for image observations."""
    env = tennis_v3.env(obs_type=obs_type)
    if obs_type == "rgb_image":
        env = ss.color_reduction_v0(env, mode="full")
        env = ss.resize_v1(env, x_size=84, y_size=84)
    return ss.frame_stack_v1(env, 4)


## Tournament

In [9]:
class AECEnvProtocol(Protocol):
    """Protocol for PettingZoo AEC environments used in tournament matches."""

    possible_agents: list[str]

    def reset(self) -> None:
        """Reset the environment to its initial state."""
        ...

    def agent_iter(self) -> Iterator[str]:
        """Iterate over agent identifiers in turn order."""
        ...

    def last(self) -> tuple[np.ndarray, float, bool, bool, dict[str, object]]:
        """Return the last transition tuple for the current agent."""
        ...

    def step(self, action: int | None) -> None:
        """Apply an action for the current agent."""
        ...


def run_tournament_match(
    env: AECEnvProtocol,
    agent1: Agent,
    agent2: Agent,
    *,
    episodes: int = 100,
    train_mode: bool = True,
    epsilon_start: float = 0.2,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.99,
    max_steps: int = 2000,
) -> tuple[dict[str, int], dict[str, list[float]]]:
    """Run a tournament match between two agents in a PettingZoo AEC environment."""
    agents_dict: dict[str, Agent] = {"first_0": agent1, "second_0": agent2}
    wins = {"first_0": 0, "second_0": 0}
    rewards_history = {"first_0": [], "second_0": []}

    current_epsilon = epsilon_start if train_mode else 0.0

    for _ in range(episodes):
        env.reset()

        previous_states: dict[str, np.ndarray | None] = dict.fromkeys(
            env.possible_agents,
            None,
        )
        previous_actions: dict[str, int | None] = dict.fromkeys(
            env.possible_agents,
            None,
        )
        episode_rewards = {"first_0": 0.0, "second_0": 0.0}

        for steps, agent_id in enumerate(env.agent_iter()):
            observation, reward, termination, truncation, _info = env.last()
            done = termination or truncation
            obs_array = np.asarray(observation)

            episode_rewards[agent_id] += reward

            action: int | None
            if done or steps >= max_steps:
                action = None
                done = True
                if reward > 0:
                    wins[agent_id] += 1
            else:
                action = agents_dict[agent_id].get_action(
                    obs_array,
                    epsilon=current_epsilon,
                )

            if train_mode:
                prev_state = previous_states[agent_id]
                prev_action = previous_actions[agent_id]
                if prev_state is not None and prev_action is not None:
                    agents_dict[agent_id].update(
                        state=prev_state,
                        action=prev_action,
                        reward=reward,
                        next_state=obs_array,
                        done=done,
                        next_action=action,
                    )

            if not done:
                previous_states[agent_id] = obs_array
                previous_actions[agent_id] = action

            env.step(action)

            if steps + 1 >= max_steps:
                break

        rewards_history["first_0"].append(episode_rewards["first_0"])
        rewards_history["second_0"].append(episode_rewards["second_0"])

        if train_mode:
            current_epsilon = max(epsilon_end, current_epsilon * epsilon_decay)

    return wins, rewards_history


def plot_learning_curves(rewards_history: dict[str, list[float]], window: int = 50) -> None:
    """Plot learning curves for each agent using a moving average of rewards."""
    plt.figure(figsize=(10, 5))

    for agent_id, rewards in rewards_history.items():
        if len(rewards) >= window:
            ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
            plt.plot(np.arange(window - 1, len(rewards)), ma, label=f"Agent {agent_id}")
        else:
            plt.plot(rewards, label=f"Agent {agent_id} (Raw)")

    plt.xlabel("Episodes")
    plt.ylabel(f"Moving Average Reward (Window={window})")
    plt.legend()
    plt.grid(visible=True)
    plt.show()


def plot_win_rate_matrix(results_matrix: np.ndarray, agent_names: list[str]) -> None:
    """Plot a heatmap of win rates between agents."""
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        results_matrix,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=agent_names,
        yticklabels=agent_names,
    )
    plt.xlabel("Opponent")
    plt.ylabel("Agent")
    plt.title("Win Rate Matrix")
    plt.show()


In [10]:
def evaluate_tournament(
    env: AECEnvProtocol,
    agents: dict[str, Agent],
    episodes_per_match: int = 10,
) -> tuple[np.ndarray, list[str]]:
    """Evaluate a round-robin tournament between agents, returning a win rate matrix and agent names."""
    agent_names = list(agents.keys())
    n_agents = len(agent_names)
    win_matrix = np.zeros((n_agents, n_agents))

    matchups = list(itertools.permutations(enumerate(agent_names), 2))
    total_matchups = len(matchups)

    for idx, ((idx1, name1), (idx2, name2)) in enumerate(matchups, start=1):
        print(f"[Evaluation {idx}/{total_matchups}] Match : {name1} vs {name2}")

        agent1 = agents[name1]
        agent2 = agents[name2]

        wins, _ = run_tournament_match(
            env=env,
            agent1=agent1,
            agent2=agent2,
            episodes=episodes_per_match,
            train_mode=False,
            epsilon_start=0.0,
        )

        total_games = wins["first_0"] + wins["second_0"]
        if total_games > 0:
            win_matrix[idx1, idx2] = wins["first_0"] / total_games

        print(f"-> Résultat : {name1} {wins['first_0']} - {wins['second_0']} {name2}\n")

    return win_matrix, agent_names


In [11]:
env = create_env(obs_type="ram")
env.reset()

obs_space = env.observation_space("first_0")
n_actions = env.action_space("first_0").n
n_features = int(np.prod(obs_space.shape))

agent_random = RandomAgent(seed=42, action_space=n_actions)
agent_q = LinearAgent(n_features=n_features, n_actions=n_actions, method="q_learning")
agent_sarsa = LinearAgent(n_features=n_features, n_actions=n_actions, method="sarsa")
agent_dqn = DQNAgent(n_features=n_features, n_actions=n_actions)
agent_mc = MonteCarloAgent(n_features=n_features, n_actions=n_actions)

agents_dict = {
    "Random": agent_random,
    "Q-Learning": agent_q,
    "SARSA": agent_sarsa,
    "DQN": agent_dqn,
    "MC": agent_mc,
}


In [ ]:
matchups = list(itertools.permutations(agents_dict.keys(), 2))
total_matchups = len(matchups)

for idx, (name1, name2) in enumerate(matchups, start=1):
    print(f"[{idx}/{total_matchups}] Début entraînement : {name1} vs {name2}")

    wins, rewards_history = run_tournament_match(
        env=env,
        agent1=agents_dict[name1],
        agent2=agents_dict[name2],
        episodes=1000,
        train_mode=True,
        epsilon_start=0.2,
    )

    plot_learning_curves(rewards_history, window=10)

    print(f"[{idx}/{total_matchups}] Fin entraînement : {name1} vs {name2}")
    if idx < total_matchups:
        print("→ Passage au match suivant...\n")
    else:
        print("→ Tous les matchs d'entraînement sont terminés.\n")

win_matrix, agent_names = evaluate_tournament(env, agents_dict, episodes_per_match=20)
plot_win_rate_matrix(win_matrix, agent_names)


[1/20] Début entraînement : Random vs Q-Learning
